# 02. 이미지 분류 (Custom Dataset)
ImageDataGenerator와 MobileNetV2를 이용한 나만의 분류기 만들기.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# 데이터셋 다운로드 (예: 고양이/강아지)
_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)
import os
PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')
train_dir = os.path.join(PATH, 'train')
val_dir = os.path.join(PATH, 'validation')

In [ ]:
# 제너레이터 (전처리 포함)
train_image_generator = ImageDataGenerator(rescale=1./255, rotation_range=40, horizontal_flip=True)
val_image_generator = ImageDataGenerator(rescale=1./255)

train_data_gen = train_image_generator.flow_from_directory(batch_size=32, directory=train_dir, shuffle=True, target_size=(150, 150), class_mode='binary')
val_data_gen = val_image_generator.flow_from_directory(batch_size=32, directory=val_dir, target_size=(150, 150), class_mode='binary')

In [ ]:
# 전이 학습 모델
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

global_average_layer = GlobalAveragePooling2D()
prediction_layer = Dense(1, activation='sigmoid') # 이진 분류

model = tf.keras.Sequential([
  base_model,
  global_average_layer,
  prediction_layer
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_data_gen, epochs=5, validation_data=val_data_gen)